In [ ]:
import pandas as pd
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Generators

from ax.core.observation import ObservationFeatures
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy
import json
import subprocess
import os
import re

In [ ]:
iteration_to_update = 5
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"



In [ ]:
ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


In [ ]:
def update_data_to_optimizer(ax_client, list_of_new_failures):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    for trial_index in list_of_new_failures:
        # Make sure we actually have this trial
        if trial_index not in trials_df["trial_index"].values:
            print(f"Trial {trial_index} not found – skipping.")
            continue

        # Build the forced-failure payload
        new_data = {
            "success": 0,
            "surfactant_input": 1,
            "complexity": 1,
        }

        # Update the trial in-place
        ax_client.update_trial_data(trial_index=trial_index, raw_data=new_data)

    ax_client.save_to_json_file(updated_path)

    print(f"Updated trial {trial_index}: set success=0, surfactant_input=1, complexity=1")

    return ax_client

In [ ]:
list_of_new_failures = []

In [ ]:
print("Please double check the results you are updating...")
print("*" * 100)
print("*" * 100)

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


In [ ]:
new_ax_client = update_data_to_optimizer(ax_to_update, list_of_new_failures)

In [ ]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials